In [2]:
# Amanda Rodgers
# Week 5 Homework
# 09/2/26

In [3]:
# IMPORT LIBRARIES
import requests
import pandas as pd
import numpy as np

In [4]:
# Download ETH data from Coingecko

# This is the URL for CoinGecko's Ethereum market data API.
url = "https://api.coingecko.com/api/v3/coins/ethereum/market_chart"


# These are the settings we send with our API request.
params = {
    "vs_currency": "usd",  # Get Bitcoin prices in US Dollars
    "days": "180",         # Get the last 180 days of data
    "interval": "daily"    # Get one data point per day
}


# requests.get() sends a request to the API.
# The API then sends data back to Python.
response = requests.get(url, params=params)


# raise_for_status() checks if the API request was successful.
# If there is an error, Python will display the error instead
# of continuing with incorrect or missing data.
response.raise_for_status()


# .json() converts the API response into Python data.
# The result is usually a dictionary containing lists.
data = response.json()

In [5]:
# Create Pandas dataframe

# The API gives us price data in this format:
#
# [timestamp, price]
#
# Example:
# [1725148800000, 59000]
#
# pd.DataFrame() converts this data into a table.
prices = pd.DataFrame(
    data["prices"],
    columns=["timestamp", "Close"]
)

# "timestamp" = the date and time in milliseconds
# "Close" = the Bitcoin price


# The API gives us volume data in this format:
#
# [timestamp, volume]
#
# We also convert this into a DataFrame.
volumes = pd.DataFrame(
    data["total_volumes"],
    columns=["timestamp", "Volume"]
)

# "Volume" represents the amount of Ethereum trading activity.

In [6]:
# Convert timestamp into real date

# pd.to_datetime() converts timestamps into readable dates.
#
# unit="ms" means the timestamp is measured in milliseconds.
prices["Date"] = pd.to_datetime(
    prices["timestamp"],
    unit="ms"
)


# We do the same thing for the volume DataFrame.
volumes["Date"] = pd.to_datetime(
    volumes["timestamp"],
    unit="ms"
)

In [ ]:
# Combine price and volume dataframes

# merge() combines two DataFrames together.
#
# We combine them using the "Date" column.
#
# on="Date" tells pandas which column to use when matching data.
df = prices.merge(
    volumes,
    on="Date"
)


# Select only the columns we want to keep.
#
# We do not need the original timestamp columns anymore.
df = df[["Date", "Close", "Volume"]]


# sort_values() puts the data in chronological order.
#
# ascending=True means oldest dates come first.
df = df.sort_values(
    "Date",
    ascending=True
)


# set_index() makes the Date column the index of the DataFrame.
#
# An index helps us work with time-series data.
df = df.set_index("Date")


# Display the first 5 rows of the original data.
#
# head() shows the first 5 rows by default.
print("RAW ETHEREUM DATA")
print(df.head())